# Zomato Dataset — Exploratory Data Analysis

## 1. Objective

The objective of this notebook is to analyze the cleaned Zomato dataset and identify patterns that can help us understand historical restaurant business performance.

This EDA is specifically designed for the **Agentic AI-Based Business Potential Prediction in a Locality** project.

The purpose is not to create a visualization-heavy dashboard, but to identify useful features, relationships, and patterns that can support the machine learning component of the system.

## 2. Project Context

The final system will evaluate the potential of starting a food and beverage business in a particular locality using:

- Historical Zomato business data
- Google Places API for current competition
- Population data for market potential
- OpenStreetMap for surrounding infrastructure and points of interest

This notebook focuses only on the **historical Zomato data**.

## 3. EDA Goals

The analysis will focus on:

- Understanding the distributions of important numerical variables
- Investigating restaurant performance indicators
- Identifying relationships between ratings, votes, and cost
- Understanding location-level differences
- Understanding restaurant-type differences
- Identifying useful features for machine learning
- Finding patterns that can help define a defensible business-performance target

## 4. Key Performance Indicators

The main historical indicators considered are:

- `rate` — customer rating
- `votes` — customer engagement
- `approx_cost_for_two_people` — price positioning
- `online_order` — online ordering availability
- `book_table` — table booking availability
- `location` — locality
- `rest_type` — restaurant/business type
- `cuisines` — cuisine category
- `listed_in_type` — business/service category
- `listed_in_city` — broader locality classification

## 5. Important Note

Restaurant rating alone will **not** be assumed to represent business success.

The analysis will investigate multiple indicators before defining the **Business Performance / Success Target** used for machine learning.

Only relevant visualizations and statistical analyses will be performed. The goal is to generate actionable insights for the ML and Agentic AI pipeline rather than create a dashboard.

## 6. Expected Outcome

By the end of this notebook, we aim to determine:

1. Which historical features are useful for prediction
2. Which variables require further transformation
3. Which business and locality characteristics are associated with stronger performance
4. How historical restaurant performance can be represented as a machine learning target

The findings will be carried forward into the feature engineering and machine learning stages.

In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv(r"C:\Users\USER\OneDrive\Desktop\Business_Analysis_Agent\data\processed\zomato_cleaned.csv")
df.shape

(51696, 11)

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 51696 entries, 0 to 51695
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   online_order               51696 non-null  str    
 1   book_table                 51696 non-null  str    
 2   rate                       41665 non-null  float64
 3   votes                      51696 non-null  int64  
 4   location                   51696 non-null  str    
 5   rest_type                  51696 non-null  str    
 6   dish_liked                 23639 non-null  str    
 7   cuisines                   51696 non-null  str    
 8   approx_costfor_two_people  51696 non-null  float64
 9   listed_intype              51696 non-null  str    
 10  listed_incity              51696 non-null  str    
dtypes: float64(2), int64(1), str(8)
memory usage: 4.3 MB


In [3]:
df.describe()

,rate,votes,approx_costfor_two_people
count,41665.000000,51696.000000,51696.000000
mean,3.700449,283.812771,554.454407
std,0.440513,803.981766,437.641522
min,1.800000,0.000000,40.000000
25%,3.400000,7.000000,300.000000
50%,3.700000,41.000000,400.000000
75%,4.000000,198.000000,650.000000
max,4.900000,16832.000000,6000.000000


In [4]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Rows: 51696
Columns: 11

Column names:
['online_order', 'book_table', 'rate', 'votes', 'location', 'rest_type', 'dish_liked', 'cuisines', 'approx_costfor_two_people', 'listed_intype', 'listed_incity']

Missing values:
rate          10031
dish_liked    28057
dtype: int64


Performance Indicator Analysis

The historical performance of a restaurant cannot be represented by a single variable.

In this section, rating and customer votes are examined together.

- `rate` represents customer satisfaction.
- `votes` represents customer engagement and the amount of customer feedback.

The relationship between these variables will be examined to determine whether they provide complementary information for the eventual business-performance target.

In [5]:
print(df[["rate", "votes"]].corr())

          rate    votes
rate   1.00000  0.43404
votes  0.43404  1.00000


In [6]:
rating_votes = (
    df.dropna(subset=["rate"])
      .groupby("rate")["votes"]
      .agg(["count", "mean", "median"])
      .reset_index()
)

display(rating_votes)

,rate,count,mean,median
0,1.8,5,224.600000,225.0
1,2.0,11,371.909091,396.0
2,2.1,24,296.500000,253.5
3,2.2,26,285.961538,406.0
4,2.3,51,184.215686,176.0
5,2.4,70,161.171429,128.5
6,2.5,101,156.653465,90.0
7,2.6,260,114.165385,68.0
8,2.7,307,113.498371,67.0
9,2.8,600,118.400000,75.0


Cost and Performance Analysis

The approximate cost for two people represents the restaurant's price positioning.

This analysis examines whether restaurant cost is associated with historical customer ratings and engagement.

The purpose is to determine whether price should be considered as an input feature for the business-performance model.

In [7]:
cost_analysis = (
    df.groupby("approx_costfor_two_people")
      .agg(
          restaurant_count=("votes", "size"),
          avg_rating=("rate", "mean"),
          median_votes=("votes", "median"),
          avg_votes=("votes", "mean")
    )
    .sort_values("restaurant_count", ascending=False)
)

display(cost_analysis.head(20))

,restaurant_count,avg_rating,median_votes,avg_votes
approx_costfor_two_people,,,,
300.0,7576,3.571120,13.0,53.441394
400.0,6887,3.614195,30.0,113.588355
500.0,4980,3.605743,45.0,150.494779
200.0,4857,3.565805,10.0,43.601812
600.0,3714,3.678100,75.0,369.351373
250.0,2959,3.550000,14.0,58.049341
800.0,2285,3.847042,195.0,449.914223
150.0,2066,3.574410,8.0,51.500484
700.0,1948,3.718281,111.0,238.074949


In [8]:
df["cost_band"] = pd.cut(
    df["approx_costfor_two_people"],
    bins=[0, 300, 600, 1000, 2000, np.inf],
    labels=[
        "Budget",
        "Mid-Range",
        "Upper-Mid",
        "Premium",
        "Luxury"
    ]
)

cost_band_analysis = (
    df.groupby("cost_band", observed=True)
      .agg(
          restaurant_count=("votes", "size"),
          avg_rating=("rate", "mean"),
          median_votes=("votes", "median"),
          avg_votes=("votes", "mean")
    )
)

display(cost_band_analysis)

,restaurant_count,avg_rating,median_votes,avg_votes
cost_band,,,,
Budget,18554,3.567165,11.0,50.108925
Mid-Range,19530,3.617817,42.0,175.748387
Upper-Mid,8332,3.804262,176.5,481.916227
Premium,4656,4.130434,684.0,1244.465206
Luxury,624,4.123501,428.0,801.830128


Location-Level Performance Analysis

Location is a central component of the business potential prediction system.

This analysis compares restaurant activity and historical performance across different localities.

The objective is to determine whether locality-level differences exist in:

- Restaurant density
- Average rating
- Customer engagement
- Price positioning

These findings will later be combined with external locality information from Google Places, population data, and OpenStreetMap.

In [9]:
location_analysis = (
    df.groupby("location")
      .agg(
          restaurant_count=("votes", "size"),
          avg_rating=("rate", "mean"),
          median_votes=("votes", "median"),
          avg_votes=("votes", "mean"),
          median_cost=("approx_costfor_two_people", "median")
      )
      .sort_values("restaurant_count", ascending=False)
)

display(location_analysis.head(20))

,restaurant_count,avg_rating,median_votes,avg_votes,median_cost
location,,,,,
BTM,5124,3.573740,23.0,120.877440,325.0
HSR,2523,3.672164,48.0,198.065795,400.0
Koramangala 5th Block,2504,4.005821,207.0,886.384185,600.0
JP Nagar,2235,3.675306,40.0,262.457718,450.0
Whitefield,2144,3.621618,24.0,217.737407,450.0
Indiranagar,2083,3.828154,134.0,574.175228,500.0
Jayanagar,1926,3.780280,76.0,253.416407,400.0
Marathahalli,1846,3.541927,29.0,241.170639,400.0
Bannerghatta Road,1630,3.507449,20.0,134.403067,400.0


 Restaurant-Type Performance Analysis

Restaurant type represents the nature of the food and beverage business.

This analysis examines whether historical performance differs across restaurant types.

The results will help determine whether `rest_type` should be included as an important feature in the machine learning model.

In [10]:
rest_type_analysis = (
    df.groupby("rest_type")
      .agg(
          restaurant_count=("votes", "size"),
          avg_rating=("rate", "mean"),
          median_votes=("votes", "median"),
          avg_votes=("votes", "mean"),
          median_cost=("approx_costfor_two_people", "median")
      )
      .sort_values("restaurant_count", ascending=False)
)

display(rest_type_analysis.head(20))

,restaurant_count,avg_rating,median_votes,avg_votes,median_cost
rest_type,,,,,
Quick Bites,19338,3.542887,17.0,77.179646,300.0
Casual Dining,10330,3.740398,158.0,440.414037,700.0
Cafe,3732,3.847342,94.0,389.813237,600.0
Delivery,2604,3.569286,12.0,85.931644,400.0
Dessert Parlor,2263,3.875483,43.0,142.844896,300.0
"Takeaway, Delivery",2037,3.513622,10.0,44.318115,400.0
"Casual Dining, Bar",1154,4.079009,509.0,1061.878683,1200.0
Bakery,1141,3.607955,8.0,31.186678,400.0
Beverage Shop,867,3.626324,13.0,34.833910,200.0


Feature Usefulness Assessment

Based on the exploratory analysis, the available variables are assessed according to their potential usefulness for the business-potential prediction model.

Features are evaluated based on:

- Business relevance
- Historical relationship with performance
- Availability for future predictions
- Potential usefulness when combined with external data sources

The objective is to identify features that should be carried forward into feature engineering.

In [11]:
feature_summary = pd.DataFrame({
    "feature": [
        "rate",
        "votes",
        "approx_costfor_two_people",
        "location",
        "rest_type",
        "cuisines",
        "online_order",
        "book_table",
        "listed_intype",
        "listed_incity"
    ],
    "role": [
        "Historical performance",
        "Historical engagement",
        "Price positioning",
        "Locality",
        "Business type",
        "Cuisine",
        "Service capability",
        "Service capability",
        "Business category",
        "City classification"
    ],
    "keep": [
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes"
    ]
})

display(feature_summary)

,feature,role,keep
0,rate,Historical performance,Yes
1,votes,Historical engagement,Yes
2,approx_costfor_two_people,Price positioning,Yes
3,location,Locality,Yes
4,rest_type,Business type,Yes
5,cuisines,Cuisine,Yes
6,online_order,Service capability,Yes
7,book_table,Service capability,Yes
8,listed_intype,Business category,Yes
9,listed_incity,City classification,Yes


 EDA Conclusion

The exploratory analysis of the cleaned Zomato dataset identified several variables that provide useful historical information about restaurant businesses.

### Key Findings

1. **Restaurant rating and customer votes show a moderate positive association**, indicating that customer satisfaction and customer engagement provide complementary performance signals.

2. **Price positioning is associated with historical restaurant performance.** Higher-cost restaurant groups generally show higher ratings and customer engagement, although the relationship is not strictly linear.

3. **Restaurant performance varies significantly across localities.** Different Bangalore locations show substantial differences in ratings, customer engagement, restaurant density, and price levels.

4. **Restaurant type is associated with different performance patterns.** Business categories such as cafes, casual dining, pubs, and fine dining show different historical levels of rating and customer engagement.

5. **Location, business type, cuisine, service capabilities, and price positioning should be retained as potential predictive features.**

6. **Rating alone is not sufficient to represent business success.** Historical performance should consider multiple indicators such as rating and customer engagement.

### Features Selected for Feature Engineering

The following features will be carried forward:

- `rate`
- `votes`
- `approx_costfor_two_people`
- `location`
- `rest_type`
- `cuisines`
- `online_order`
- `book_table`
- `listed_intype`
- `listed_incity`

### Important Limitation

The Zomato dataset represents historical restaurant information. It does not directly represent current market conditions such as present-day competition, population, or surrounding infrastructure.

Therefore, Zomato data alone will not be used to make the final business recommendation.

Instead, it will form the **historical business-performance layer** of the system and will later be combined with:

- Google Places API → Current competition
- Population data → Market potential
- OpenStreetMap → Local infrastructure and nearby points of interest

### Next Step

The next stage is **Feature Engineering**, where the selected variables will be transformed into structured numerical features suitable for machine learning and integration with the external data sources.

**Status: Zomato EDA Completed**